In [0]:
verification_paths = {
    # Silver
    "silver_employees": (
        "abfss://silver@stnenimadlsdev01.dfs.core.windows.net/"
        "northstar/employees"
    ),
    "silver_dependents": (
        "abfss://silver@stnenimadlsdev01.dfs.core.windows.net/"
        "northstar/dependents"
    ),
    "silver_enrollments": (
        "abfss://silver@stnenimadlsdev01.dfs.core.windows.net/"
        "northstar/enrollments"
    ),
    "silver_eligibility": (
        "abfss://silver@stnenimadlsdev01.dfs.core.windows.net/"
        "northstar/eligibility"
    ),

    # Gold employee analytics
    "gold_employee_summary": (
        "abfss://gold@stnenimadlsdev01.dfs.core.windows.net/"
        "northstar/employee_summary"
    ),
    "gold_department_summary": (
        "abfss://gold@stnenimadlsdev01.dfs.core.windows.net/"
        "northstar/department_summary"
    ),
    "gold_employer_summary": (
        "abfss://gold@stnenimadlsdev01.dfs.core.windows.net/"
        "northstar/employer_summary"
    ),
    "gold_state_summary": (
        "abfss://gold@stnenimadlsdev01.dfs.core.windows.net/"
        "northstar/state_summary"
    ),
    "gold_executive_kpis": (
        "abfss://gold@stnenimadlsdev01.dfs.core.windows.net/"
        "northstar/executive_kpis"
    ),

    # Gold reconciliation
    "gold_eligibility_reconciliation": (
        "abfss://gold@stnenimadlsdev01.dfs.core.windows.net/"
        "northstar/eligibility_reconciliation"
    ),
    "gold_eligibility_reconciliation_kpis": (
        "abfss://gold@stnenimadlsdev01.dfs.core.windows.net/"
        "northstar/eligibility_reconciliation_kpis"
    ),
}

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
)

verification_results = []

for dataset_name, path in verification_paths.items():
    try:
        df = spark.read.format("delta").load(path)
        row_count = df.count()

        verification_results.append(
            (
                dataset_name,
                path,
                "PASS",
                row_count,
                None,
            )
        )

    except Exception as exc:
        verification_results.append(
            (
                dataset_name,
                path,
                "FAIL",
                None,
                str(exc)[:500],
            )
        )

verification_schema = StructType(
    [
        StructField("dataset_name", StringType(), False),
        StructField("path", StringType(), False),
        StructField("status", StringType(), False),
        StructField("row_count", LongType(), True),
        StructField("error_message", StringType(), True),
    ]
)

verification_df = spark.createDataFrame(
    verification_results,
    schema=verification_schema,
)

display(
    verification_df.orderBy("dataset_name")
)

In [0]:
display(
    verification_df.select(
        "dataset_name",
        "status",
        "row_count",
    ).orderBy("dataset_name")
)

In [0]:
for dataset_name, path in verification_paths.items():
    print("=" * 100)
    print(dataset_name)
    print(path)

    try:
        df = spark.read.format("delta").load(path)
        df.printSchema()
        display(df.limit(5))
    except Exception as exc:
        print(f"FAILED: {exc}")

In [0]:
METRICS_PATH = (
    "abfss://gold@stnenimadlsdev01.dfs.core.windows.net/"
    "northstar/data_quality_metrics"
)

metrics_df = (
    spark.read
    .format("delta")
    .load(METRICS_PATH)
)

display(
    metrics_df.orderBy(
        F.col("run_timestamp").desc()
    )
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

latest_window = (
    Window
    .partitionBy("pipeline_name")
    .orderBy(F.col("run_timestamp").desc())
)

latest_metrics_df = (
    metrics_df
    .withColumn(
        "row_number",
        F.row_number().over(latest_window),
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

print("Latest metrics DataFrame rebuilt successfully.")

In [0]:
display(
    latest_metrics_df.orderBy("pipeline_name")
)

In [0]:
gold_root = (
    "abfss://gold@stnenimadlsdev01.dfs.core.windows.net/"
    "northstar"
)

display(dbutils.fs.ls(gold_root))

In [0]:
candidate_paths = [
    f"{gold_root}/employee_summary",
    f"{gold_root}/state_summary",
]

for path in candidate_paths:
    try:
        count = spark.read.format("delta").load(path).count()
        print(f"PASS | {count:,} | {path}")
    except Exception as exc:
        print(f"FAIL | {path}")
        print(str(exc)[:300])